In [1]:
import torch
import numpy as np
import uproot
import awkward as ak
import matplotlib.pyplot as plt
import sys
sys.path.append('../utils/')
sys.path.append('..')
import plotting_utils as pu
import data_utils as du
from of_dataset import OfDataset
from lightning_module import LOfTransformer

In [2]:
# Get data and pseudodata files
mc_file = uproot.open('/global/cfs/cdirs/m3246/ZjetOmnifold/data/slimmed_files/WithTracks_ZjetOmnifold_May19_MGPy8FxFxRew_syst_train_Mar1023.root')
pd_file = uproot.open('/global/cfs/cdirs/m3246/ZjetOmnifold/data/slimmed_files/WithTracks_ZjetOmnifold_Aug5_PseudoDataSRew_Apr8_1_All.root')
mc_tree = mc_file['OmniTree']
pd_tree = pd_file['OmniTree']

In [3]:
# Get pass190s
mc_pass190 = ak.to_numpy(mc_tree['pass190'].array())
pd_pass190 = ak.to_numpy(pd_tree['pass190'].array())

In [4]:
# Get kinematics and one hots for MC and PD
mc_kinematics, mc_indeces = du.get_kinematics(mc_tree, filter=mc_pass190)
pd_kinematics, pd_indeces = du.get_kinematics(pd_tree, filter=pd_pass190)

/global/homes/k/kgreif/.conda/envs/zjets/lib/python3.11/site-packages/awkward/_nplikes/array_module.py:245: RuntimeWarning: invalid value encountered in log
  return impl(*broadcasted_args, **(kwargs or {}))


In [5]:
# Make weights
mc_weights = np.ones(len(mc_kinematics))
pd_weights = np.ones(len(pd_kinematics))

In [6]:
# Make labels
mc_labels = np.zeros((len(mc_kinematics), 1), dtype=np.int32)
pd_labels = np.ones((len(pd_kinematics), 1), dtype=np.int32)

In [7]:
# Make dummy plotting
mc_plotting = np.zeros((len(mc_kinematics), 1), dtype=np.int32)
pd_plotting = np.ones((len(pd_kinematics), 1), dtype=np.int32)

In [8]:
# Concatenate all of the data
kinematics = ak.concatenate((mc_kinematics, pd_kinematics), axis=0)
indeces = ak.concatenate((mc_indeces, pd_indeces), axis=0)
weights = np.concatenate((mc_weights, pd_weights), axis=0)
labels = np.concatenate((mc_labels, pd_labels), axis=0)
plotting = np.concatenate((mc_plotting, pd_plotting), axis=0)

In [9]:
print(len(kinematics))
print(len(indeces))
print(len(weights))
print(len(labels))
print(len(plotting))

1407402
1407402
1407402
1407402
1407402


In [10]:
# Make pytorch datasets
all_dataset = OfDataset(kinematics, labels, weights, plotting, object_indeces=indeces, max_tracks=264)

In [11]:
# Do train / val split
generator = torch.Generator().manual_seed(331602)
train_dataset, val_dataset = torch.utils.data.random_split(all_dataset, [0.8, 0.2], generator=generator)

In [12]:
# Print the muon pT for the first event in both sets
train_muon_pt = train_dataset[0][0][0,0,:2]
val_muon_pt = val_dataset[0][0][0,0,:2]
print("Train muon pt: {} {}".format(train_muon_pt[0], train_muon_pt[1]))
print("Val muon pt: {} {}".format(val_muon_pt[0], val_muon_pt[1]))

Train muon pt: 4.793519496917725 4.425823211669922
Val muon pt: 5.282020092010498 4.494627952575684


In [19]:
# Load trained model
model = LOfTransformer.load_from_checkpoint('../checkpoints/zjets-unfolding/y71nctdg/checkpoints/epoch=49-val_loss=0.4225.ckpt')
model.cpu()
model.eval()

Batch normalization disabled in embeddings!


LOfTransformer(
  (criterion): BCEWithLogitsLoss()
  (model): OfTransformer(
    (trimmer): SequenceTrimmer()
    (embed): Embed(
      (embed): InputDistributed(
        (module): Sequential(
          (0): Linear(in_features=11, out_features=128, bias=True)
          (1): GELU(approximate='none')
          (2): Linear(in_features=128, out_features=512, bias=True)
          (3): GELU(approximate='none')
          (4): Linear(in_features=512, out_features=128, bias=True)
          (5): GELU(approximate='none')
        )
      )
    )
    (pair_embed): PairEmbed(
      (embed): Sequential(
        (0): Conv1d(4, 64, kernel_size=(1,), stride=(1,))
        (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): GELU(approximate='none')
        (3): Conv1d(64, 64, kernel_size=(1,), stride=(1,))
        (4): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (5): GELU(approximate='none')
        (6): Conv1d(64, 6

In [24]:
all_label = all_dataset[1][1]
all_output = model(all_dataset[1][0], all_dataset[1][2])

/global/homes/k/kgreif/.conda/envs/zjets/lib/python3.11/site-packages/torch/nn/functional.py:5076: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(


In [25]:
print(all_output)

tensor([[0.4085]], grad_fn=<AddmmBackward0>)
